# CLM-001-QFIRST — integrated Q-first training

Default: **CPU Gate 0/1 only**. No GPU training is authorized by this Notebook.

Reuse the original `bingjiangli/minicells` Notebook and its existing `GITHUB_TOKEN` / `HF_TOKEN` Secrets. Run All performs the selected phase and returns whitelisted metadata to GitHub. Full Checkpoints remain in saved Kaggle Output.

GPU Pilot and 50/500-step training each require the separate review and frozen approval configuration specified in SPEC.md. A passed CPU Gate cannot grant those approvals.


In [ ]:
from pathlib import Path

MODE = "cpu-preflight"  # cpu-preflight | gpu-pilot | train-smoke | train-dev | publish-existing
SOURCE_REF = "codex/clm001-qfirst-integrated"  # resolved once to an immutable SHA during setup
APPROVAL_CONFIG = (
    None  # supplied only after the separate protocol review; never toggle gate flags here
)
RETURN_TO_GITHUB = True
ROOT = Path("/kaggle/working/clm001-qfirst-code")
OUTPUT = Path("/kaggle/working/clm001-qfirst-output")
ASSETS = Path("/kaggle/working/clm001-qfirst-assets")

In [ ]:
import subprocess
import sys


def run(args, **kwargs):
    print("[CLM-001]", " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), check=True, **kwargs)


# Local paths are native Kaggle paths; never rewrite them to /shared/kaggle.
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "pytest",
        "numpy",
        "tokenizers==0.20.3",
        "huggingface_hub",
        "datasets>=2.18,<5",
    ]
)
if not (ROOT / ".git").exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init", "-b", "clm001", ROOT])
    run(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], cwd=ROOT)
    run(["git", "sparse-checkout", "init", "--no-cone"], cwd=ROOT)
    run(
        [
            "git",
            "sparse-checkout",
            "set",
            "--no-cone",
            "/pyproject.toml",
            "/src/",
            "/shared/python/",
            "/tests/research/clm001_qfirst/",
            "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
            "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
            "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
            "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
            "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
        ],
        cwd=ROOT,
    )
run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_REF], cwd=ROOT)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT)
source = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
BASE = ROOT / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated"
OUTPUT = OUTPUT / source[:12]
OUTPUT.mkdir(parents=True, exist_ok=True)
config = Path(APPROVAL_CONFIG) if APPROVAL_CONFIG else BASE / "config/proposed.json"
print(
    {"experiment": "CLM-001-QFIRST", "mode": MODE, "source": source, "scientific_decision": False},
    flush=True,
)


def command(mode, output=OUTPUT, extra=()):
    return [
        sys.executable,
        BASE / "run.py",
        "--mode",
        mode,
        "--config",
        config,
        "--output",
        output,
        *extra,
    ]


if MODE == "publish-existing":
    run(command("publish"), cwd=ROOT)
else:
    run(command("cpu-preflight"), cwd=ROOT)
    if MODE != "cpu-preflight":
        if APPROVAL_CONFIG is None:
            raise RuntimeError(
                "CPU Gate passed. GPU Pilot/development remain unapproved; use the reviewed approval configuration."
            )
        # Existing publisher checks write access before any GPU computation.
        run(command("publish-preflight"), cwd=ROOT)
        if not ASSETS.exists():
            run(command("prepare", extra=("--assets", ASSETS)), cwd=ROOT)
        if MODE == "gpu-pilot":
            run(command(MODE, extra=("--assets", ASSETS, "--device", "cuda:0")), cwd=ROOT)
        elif MODE in ("train-smoke", "train-dev"):
            children = []
            for index, arm in enumerate(("QF-FULL", "J0-FULL")):
                args = command(
                    MODE,
                    OUTPUT / arm,
                    ("--assets", ASSETS, "--device", f"cuda:{index}", "--arm", arm),
                )
                children.append(subprocess.Popen(list(map(str, args)), cwd=ROOT))
            codes = [child.wait() for child in children]
            if any(codes):
                raise RuntimeError(f"Development failed: {codes}; retained Checkpoints can resume.")
            run(command("analyze"), cwd=ROOT)
        else:
            raise ValueError("Unknown phase")
    if RETURN_TO_GITHUB:
        run(command("publish"), cwd=ROOT)
print("Retain with Save Version:", OUTPUT, flush=True)